# VecDB Advanced Search & Diagnostics
Navigate advanced filtering, diagnostics, and error scenarios using the Oracle VecDB Python SDK.

## 1. Scenario Overview
- Load VecDB credentials, create a reusable client, and seed demo rows.
- Run semantic search with range filters and logical channels filters.
- Enable debug flags to inspect query profiles.
- Capture API errors gracefully for resilient integrations.

## Architecture at a Glance

```text
Search Diagnostics Matrix

                            +-----------------------------+
                            | SEARCH_ADV_DEMO seed table  |
                            | title, audience, priority,  |
                            | channels, deterministic vec |
                            +--------------+--------------+
                                           |
        +----------------------------------+----------------------------------+
        |                                  |                                  |
        v                                  v                                  v
+------------------+              +------------------+              +------------------+
| Range filter     |              | Logical filter   |              | Debug/error path |
+------------------+              +------------------+              +------------------+
| PRIORITY >= 80   |              | OR over audience |              | debug_flags      |
| top_k=5          |              | and channels     |              | missing table    |
+--------+---------+              +--------+---------+              +--------+---------+
         |                                 |                                 |
         v                                 v                                 v
+------------------+              +------------------+              +------------------+
| compare returned |              | inspect matched  |              | confirm payload  |
| titles/distances |              | metadata fields  |              | or caught error  |
+--------+---------+              +--------+---------+              +--------+---------+
         |                                 |                                 |
         +-----------------------+---------+---------------------------------+
                                 |
                                 v
                    +-----------------------------+
                    | cleanup SEARCH_ADV_DEMO     |
                    +-----------------------------+
```

This notebook is a diagnostics playbook: seed controlled data, run filter variants, inspect returned payloads, then prove errors are caught cleanly.


## What to Validate

- The seeded table should contain deterministic metadata and vectors for repeatable diagnostics.
- Range, equality, `AND`, and `OR` filters should change which rows are returned, not the table contents.
- The error-handling section should catch the missing-table exception cleanly and continue to cleanup.


In [ ]:
%pip install -U oracle-vecdb python-dotenv pandas


In [ ]:
import os
from dotenv import load_dotenv
from oracle_vecdb import OracleVecDB, Configuration

print('Loading environment variables for VecDB connection...')
load_dotenv()

resolved_host = os.getenv('VECDB_REST_URL')
resolved_user = os.getenv('VECDB_USERNAME') or os.getenv('VECDB_USER')
resolved_password = os.getenv('VECDB_PASSWORD')
resolved_access_token = os.getenv('VECDB_ACCESS_TOKEN')
print(f'Resolved REST endpoint: {resolved_host}')
print(f'Resolved user: {resolved_user}')

config_kwargs = {"rest_url": resolved_host}
if resolved_access_token:
    config_kwargs["access_token"] = resolved_access_token
else:
    config_kwargs["username"] = resolved_user
    config_kwargs["password"] = resolved_password
config = Configuration(**config_kwargs)
if os.getenv('VECDB_SELF_SIGNED_SSL', 'false').lower() == 'true':
    config.verify_ssl = False
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print('Disabled SSL verification for self-signed certificates.')

vecdb = OracleVecDB(config)
auth_method = 'bearer token' if config.access_token else 'username/password'
print('Connected to', config.rest_url)
print('Auth method:', auth_method)
print('VecDB client ready for search diagnostics.')

## 2. Seed Search Data
Create an annotated table with marketing collateral and push synthetic embeddings for testing.

In [ ]:
def query_items(response):
    if isinstance(response, list):
        return response
    if isinstance(response, tuple):
        return list(response)
    return getattr(response, "items", None) or getattr(response, "matches", None) or []


def result_metadata(item):
    return item.get("metadata", {}) if isinstance(item, dict) else getattr(item, "metadata", {})


def result_distance(item):
    if isinstance(item, dict):
        return item.get("distance")
    return getattr(item, "distance", getattr(item, "score", None))


def result_id(item):
    return item.get("id") if isinstance(item, dict) else getattr(item, "id", None)


def result_vector(item):
    if isinstance(item, dict):
        return item.get("vector") or item.get("dense_vector")
    return getattr(item, "vector", getattr(item, "dense_vector", None))


def result_text(item):
    return item.get("text", "") if isinstance(item, dict) else getattr(item, "text", "")


from uuid import uuid4
from random import Random
import pandas as pd

print('Preparing search diagnostics demo table...')
SEARCH_TABLE = os.getenv('SEARCH_TABLE') or 'SEARCH_ADV_DEMO'

vecdb.create_vector_table(
    name=SEARCH_TABLE,
    comment='Advanced search diagnostics table',
    annotations={'TITLE': 'string', 'AUDIENCE': 'string', 'PRIORITY': 'int', 'CHANNELS': 'string'},
    index_params={
        "metadata_index_params": {
            "auto_index": True,
            "include_paths": ["AUDIENCE", "PRIORITY", "CHANNELS"],
        },
    },
)
print(f'Created table {SEARCH_TABLE} with search-oriented metadata annotations.')

docs = [
    {'title': 'Executive webinar on analytics adoption', 'audience': 'leadership', 'priority': 95, 'channels': 'webinar,email'},
    {'title': 'Retail partner onboarding guide', 'audience': 'partners', 'priority': 70, 'channels': 'docs'},
    {'title': 'Customer success playbook', 'audience': 'cs', 'priority': 85, 'channels': 'email'},
    {'title': 'Public press release on acquisition', 'audience': 'public', 'priority': 60, 'channels': 'press'},
    {'title': 'Security incident response outline', 'audience': 'security', 'priority': 98, 'channels': 'pager'},
    {'title': 'Global marketing newsletter', 'audience': 'marketing', 'priority': 75, 'channels': 'email'},
    {'title': 'Support escalation SOP', 'audience': 'support', 'priority': 90, 'channels': 'docs'},
    {'title': 'HR onboarding checklist', 'audience': 'hr', 'priority': 55, 'channels': 'docs'},
    {'title': 'Partner webinar invitation', 'audience': 'partners', 'priority': 80, 'channels': 'webinar,email'},
    {'title': 'Finance compliance training', 'audience': 'finance', 'priority': 88, 'channels': 'webinar'}
]

def make_vec(text, dim=10):
    rng = Random(text)
    return [rng.random() for _ in range(dim)]

payload = [
    {
        'id': str(uuid4()),
        'dense_vector': make_vec(doc['title']),
        'metadata': {
            'TITLE': doc['title'],
            'AUDIENCE': doc['audience'],
            'PRIORITY': doc['priority'],
            'CHANNELS': doc['channels'],
        }
    }
    for doc in docs
]
vecdb.upsert_vectors(table_name=SEARCH_TABLE, vectors=payload)
print(f'Seeded {len(payload)} rows into {SEARCH_TABLE}.')
print('Previewing metadata payload below.')
pd.DataFrame([entry['metadata'] for entry in payload])

## 3. Vector Query with Range Filter
Apply a minimum priority threshold while performing cosine search.

In [ ]:
print('Running vector query with priority filter >= 80...')
query_vec = make_vec('analytics leaders')
results = vecdb.query(
    table_name=SEARCH_TABLE,
    query_by={'vector': query_vec},
    filters={'$and': [{'PRIORITY': {'$gte': 80}}]},
    include_vectors=False,
    top_k=5
)
items = query_items(results)
print(f'Returned {len(items)} high-priority matches.')
pd.DataFrame([
    {
        'TITLE': result_metadata(r).get('TITLE'),
        'AUDIENCE': result_metadata(r).get('AUDIENCE'),
        'PRIORITY': result_metadata(r).get('PRIORITY'),
        'distance': result_distance(r)
    }
    for r in items
])

## 4. `$or` Channel Filter
Demonstrate logical filters combining audience and channel metadata.

In [ ]:
print('Evaluating OR filter combining audience and channels...')
or_results = vecdb.query(
    table_name=SEARCH_TABLE,
    query_by={'vector': make_vec('webinar outreach')},
    filters={'$or': [
        {'AUDIENCE': {'$eq': 'partners'}},
        {'CHANNELS': {'$eq': 'webinar,email'}},
    ]},
    include_vectors=False,
    top_k=5
)
or_items = query_items(or_results)
print(f'Returned {len(or_items)} records matching OR conditions.')
pd.DataFrame([
    {
        'TITLE': result_metadata(r).get('TITLE'),
        'AUDIENCE': result_metadata(r).get('AUDIENCE'),
        'CHANNELS': result_metadata(r).get('CHANNELS')
    }
    for r in or_items
])

### Filter Playground Summary

This optional cell runs additional live VecDB queries against the same seeded table. It is intentionally left without saved output here, because the returned rows and distances should come from the environment where the notebook is run.


In [ ]:
import json
from IPython.display import display

filter_cases = [
    {
        'case': 'Range filter',
        'query': 'analytics leaders',
        'filter': {'PRIORITY': {'$gte': 80}},
    },
    {
        'case': 'Equality filter',
        'query': 'partner outreach',
        'filter': {'AUDIENCE': {'$eq': 'partners'}},
    },
    {
        'case': 'AND filter',
        'query': 'high priority webinar',
        'filter': {
            '$and': [
                {'PRIORITY': {'$gte': 80}},
                {'CHANNELS': {'$eq': 'webinar,email'}},
            ]
        },
    },
    {
        'case': 'OR filter',
        'query': 'webinar outreach',
        'filter': {
            '$or': [
                {'AUDIENCE': {'$eq': 'partners'}},
                {'CHANNELS': {'$eq': 'webinar,email'}},
            ]
        },
    },
]

comparison_rows = []
for scenario in filter_cases:
    matches = vecdb.query(
        table_name=SEARCH_TABLE,
        query_by={'vector': make_vec(scenario['query'])},
        filters=scenario['filter'],
        include_vectors=False,
        top_k=5,
    )
    top = matches[0] if matches else {'metadata': {}, 'distance': None}
    metadata = top.get('metadata', {}) if isinstance(top, dict) else getattr(top, 'metadata', {})
    distance = top.get('distance') if isinstance(top, dict) else getattr(top, 'distance', None)
    comparison_rows.append({
        'case': scenario['case'],
        'query': scenario['query'],
        'filter': json.dumps(scenario['filter']),
        'returned_rows': len(matches),
        'top_title': metadata.get('TITLE'),
        'top_audience': metadata.get('AUDIENCE'),
        'top_priority': metadata.get('PRIORITY'),
        'top_distance': distance,
    })

filter_summary = pd.DataFrame(comparison_rows)
display(filter_summary)


## 5. Debug Flags
Inspect query execution diagnostics when backend flags are available.

In [ ]:
print('Requesting query diagnostics via debug_flags...')
debug_resp = vecdb.query(
    table_name=SEARCH_TABLE,
    query_by={'vector': make_vec('analytics webinar')},
    debug_flags={'profile': 'true'},
    top_k=3
)
print('Debug response payload:')
debug_resp

## 6. Error Handling
Intentionally call a missing table to show structured exception handling.

In [ ]:
from oracle_vecdb.vecdb_errors import VecDBError

print('Demonstrating error handling by querying a missing table...')
try:
    vecdb.query(table_name='NON_EXISTENT_TABLE', query_by={'vector': [0.1, 0.2]}, top_k=1)
except VecDBError as exc:
    print('Caught VecDBError:', exc)
except Exception as exc:
    print('Caught VecDB client exception:', type(exc).__name__)
    print('Message:', exc)

## 7. Cleanup
Drop the demo table so repeated runs stay idempotent.

In [ ]:
print('Cleaning up search diagnostics table...')
vecdb.drop_vector_table(name=SEARCH_TABLE)
print(f'Removed table {SEARCH_TABLE}.')